# 🩺 Women's Health Risk Predictor
## Breast Cancer Classification using Machine Learning

> **Dataset**: Wisconsin Breast Cancer Dataset (UCI Machine Learning Repository)  
> **Goal**: Predict whether a breast mass is Benign (2) or Malignant (4) based on 9 clinical cell-level features.

### Notebook Outline
1. Data Loading & Exploration (EDA)
2. Data Preprocessing
3. Model Training (5 classifiers)
4. Model Evaluation & Comparison
5. Feature Importance
6. Model Saving

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Style settings
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 100,
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'text.color': '#e6edf3',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'axes.edgecolor': '#30363d',
    'grid.color': '#21262d',
    'axes.titlecolor': '#e6edf3',
})

RANDOM_STATE = 42
print('✅ Libraries imported successfully')

In [ ]:
# Load dataset
df = pd.read_csv('../data/dataset.csv')

print(f'Shape: {df.shape}')
print(f'\nColumn names: {list(df.columns)}')
df.head(10)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic stats
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
# Check for missing values (bare_nucleoli has '?' values)
print('=== Missing / Non-numeric Values ===')
print(f"bare_nucleoli '?' count: {(df['bare_nucleoli'] == '?').sum()}")
print(f"Total rows: {len(df)}")
print(f"Rows with '?': {(df == '?').any(axis=1).sum()}")

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
class_counts = df['class'].value_counts()
bars = axes[0].bar(
    ['Benign (2)', 'Malignant (4)'],
    [class_counts[2], class_counts[4]],
    color=['#10b981', '#ef4444'],
    alpha=0.85, edgecolor='none', width=0.6
)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold', pad=12)
axes[0].set_ylabel('Count')
for bar, cnt in zip(bars, [class_counts[2], class_counts[4]]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(cnt), ha='center', fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    [class_counts[2], class_counts[4]],
    labels=['Benign', 'Malignant'],
    colors=['#10b981', '#ef4444'],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': 'none', 'linewidth': 0}
)
for at in autotexts: at.set_color('white'); at.set_fontweight('bold')
axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold', pad=12)

plt.tight_layout()
plt.show()
print(f'Benign: {class_counts[2]} ({class_counts[2]/len(df)*100:.1f}%)')
print(f'Malignant: {class_counts[4]} ({class_counts[4]/len(df)*100:.1f}%)')

In [ ]:
# Feature distributions by class
feature_cols = [
    'clump_thickness', 'size_uniformity', 'shape_uniformity',
    'marginal_adhesion', 'epithelial_size', 'bare_nucleoli',
    'bland_chromatin', 'normal_nucleoli', 'mitoses'
]

# Temporarily convert for plotting
df_plot = df.copy()
df_plot['bare_nucleoli'] = pd.to_numeric(df_plot['bare_nucleoli'], errors='coerce')
df_plot['class_label'] = df_plot['class'].map({2: 'Benign', 4: 'Malignant'})

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()
colors = {'Benign': '#10b981', 'Malignant': '#ef4444'}

for i, col in enumerate(feature_cols):
    for label, grp in df_plot.groupby('class_label'):
        axes[i].hist(grp[col].dropna(), bins=10, alpha=0.7,
                     color=colors[label], label=label, edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value (1-10)')
    axes[i].set_ylabel('Count')
    if i == 0: axes[i].legend()

plt.suptitle('Feature Distributions by Class', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
df_corr = df_plot[feature_cols + ['class']].copy()
df_corr['class'] = df_corr['class'].map({2: 0, 4: 1})

corr_matrix = df_corr.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='#21262d',
    ax=ax, annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Clean the dataset
df_clean = df.copy()

# Handle bare_nucleoli: convert '?' to NaN, fill with median
df_clean['bare_nucleoli'] = pd.to_numeric(df_clean['bare_nucleoli'], errors='coerce')
median_bn = df_clean['bare_nucleoli'].median()
df_clean['bare_nucleoli'] = df_clean['bare_nucleoli'].fillna(median_bn)
print(f"Filled {(df['bare_nucleoli'] == '?').sum()} missing bare_nucleoli values with median={median_bn}")

# Drop ID column (not a feature)
df_clean.drop(columns=['id'], inplace=True)

# Encode target: 2 → 0 (Benign), 4 → 1 (Malignant)
df_clean['class'] = df_clean['class'].map({2: 0, 4: 1})

print(f'\nCleaned shape: {df_clean.shape}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
print(f'\nClass balance:')
print(df_clean['class'].value_counts())
df_clean.head()

In [ ]:
# Prepare features and target
X = df_clean[feature_cols].values
y = df_clean['class'].values

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')
print(f'\nClass distribution in test set:')
print(f'  Benign: {(y_test==0).sum()} | Malignant: {(y_test==1).sum()}')

## 4. Model Training

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'SVM':                 SVC(probability=True, random_state=RANDOM_STATE),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=RANDOM_STATE),
}

# Train all models
trained = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained[name] = model
    print(f'✅ Trained: {name}')

print('\n🏁 All models trained!')

## 5. Model Evaluation

In [ ]:
# Evaluate all models
results = []
for name, model in trained.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    
    results.append({
        'Model': name,
        'Test Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1 Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4),
        'CV Mean Acc': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4),
    })

df_results = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
df_results.style.background_gradient(subset=['Test Accuracy', 'ROC-AUC'], cmap='Greens').format({'CV Std': '±{:.4f}'})

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')

metrics = ['Test Accuracy', 'F1 Score', 'ROC-AUC']
colors = sns.color_palette('viridis', len(df_results))

for ax, metric in zip(axes, metrics):
    bars = ax.barh(df_results['Model'], df_results[metric], color=colors, alpha=0.85, edgecolor='none')
    ax.set_xlabel(metric)
    ax.set_title(metric, fontweight='bold')
    ax.set_xlim(0.8, 1.02)
    for bar, val in zip(bars, df_results[metric]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves for all models
fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = sns.color_palette('tab10', len(trained))

for (name, model), color in zip(trained.items(), colors_roc):
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2.5, label=f'{name}  (AUC = {auc:.4f})', color=color)

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Random Classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Classifiers', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.3)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for best model
best_model_name = df_results.iloc[0]['Model']
best_model = trained[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

print(f'Best Model: {best_model_name}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Benign', 'Malignant']))

cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign', 'Malignant'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Random Forest feature importance
rf_model = trained['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors_feat = ['#ef4444' if v >= importances.median() else '#6366f1' for v in importances]
importances.plot(kind='barh', ax=ax, color=colors_feat, alpha=0.85, edgecolor='none')
ax.axvline(importances.median(), color='#f59e0b', linestyle='--', alpha=0.7, lw=2, label=f'Median = {importances.median():.4f}')
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
ax.set_yticklabels([l.replace('_', ' ').title() for l in importances.index])
ax.legend()

# Add value labels
for i, (val, label) in enumerate(zip(importances, importances.index)):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\nTop 3 most predictive features:')
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f'  {feat.replace("_", " ").title()}: {imp:.4f}')

In [ ]:
# Logistic Regression coefficients (interpretable)
lr_model = trained['Logistic Regression']
lr_coefs = pd.Series(lr_model.coef_[0], index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
colors_coef = ['#ef4444' if c > 0 else '#10b981' for c in lr_coefs]
lr_coefs.plot(kind='barh', ax=ax, color=colors_coef, alpha=0.85, edgecolor='none')
ax.axvline(0, color='white', linewidth=0.8, alpha=0.4)
ax.set_xlabel('Coefficient Value (positive → Malignant)', fontsize=11)
ax.set_title('Logistic Regression Coefficients (Scaled Features)', fontsize=13, fontweight='bold')
ax.set_yticklabels([l.replace('_', ' ').title() for l in lr_coefs.index])
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
import json

# Create models directory
os.makedirs('../models', exist_ok=True)

# Save best model + scaler
bundle = {'model': best_model, 'scaler': scaler, 'features': feature_cols}
model_path = '../models/breast_cancer_model.pkl'
joblib.dump(bundle, model_path)
print(f'✅ Model saved to: {model_path}')

# Save LR coefficients as JSON for the web app
coeffs = {
    'coef': lr_model.coef_[0].tolist(),
    'intercept': float(lr_model.intercept_[0]),
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'features': feature_cols,
    'best_model': best_model_name,
    'best_accuracy': float(df_results.iloc[0]['Test Accuracy']),
    'best_roc_auc': float(df_results.iloc[0]['ROC-AUC']),
    'feature_importance': rf_model.feature_importances_.tolist(),
}
coef_path = '../models/model_coefficients.json'
with open(coef_path, 'w') as f:
    json.dump(coeffs, f, indent=2)
print(f'✅ Coefficients saved to: {coef_path}')

print(f'\n📊 Best Model Summary:')
print(df_results.iloc[0].to_string())

In [ ]:
# Test loading the saved model
loaded = joblib.load('../models/breast_cancer_model.pkl')
loaded_model = loaded['model']
loaded_scaler = loaded['scaler']

# Quick sanity check prediction
sample_benign    = np.array([[1, 1, 1, 1, 2, 1, 3, 1, 1]])  # Typical benign
sample_malignant = np.array([[8, 10, 10, 8, 7, 10, 9, 7, 1]])  # Typical malignant

for name, sample in [('Benign', sample_benign), ('Malignant', sample_malignant)]:
    scaled = loaded_scaler.transform(sample)
    pred = loaded_model.predict(scaled)[0]
    prob = loaded_model.predict_proba(scaled)[0]
    label = 'Benign' if pred == 0 else 'Malignant'
    confidence = max(prob)
    print(f'Expected {name}: → Predicted {label} ({confidence:.1%} confidence) ✅' 
          if name == label else f'Expected {name}: → Predicted {label} ({confidence:.1%} confidence) ⚠️')

print('\n🎉 Model is working correctly!')